# Fine-Tuning Genomic Models for Sequence Classification

**Day 2 Morning - Session 1**

**Author:** Ikram Ullah, KAUST Bioinformatics Platform

---

## Overview

In this notebook, we'll **fine-tune** a pre-trained Nucleotide Transformer for a real genomic task: **promoter detection**.

This is the complete workflow you'll use in your research:

1. Load a pre-trained foundation model
2. Add a classification head
3. Prepare and tokenize your dataset
4. Train using Hugging Face Trainer
5. Evaluate on held-out test data

---

## Learning Objectives

1. Build a classifier on top of a pre-trained encoder
2. Use the Hugging Face `Trainer` for training
3. Implement proper evaluation metrics
4. Interpret results with confusion matrices

---

## The Task: Promoter Detection

### What are Promoters?

**Promoters** are specific DNA sequences located **upstream** (5' direction) of genes that serve as the "launch pad" for transcription. They are where the transcription machinery assembles to begin copying DNA into RNA.

```
Gene Structure (simplified):
                                          
    5' ──────[PROMOTER]──[TSS]──────[GENE BODY]────── 3'
              ↑           ↑
         -200 to -50   Start site
         (upstream)     (position 0)
```

### Key Features of Promoters

| Feature | Description |
|---------|-------------|
| **Location** | Typically 100-1000 bp upstream of the transcription start site (TSS) |
| **Core elements** | TATA box (~-25 bp), Initiator (Inr), TFIIB recognition element |
| **Function** | Recruit RNA polymerase II and transcription factors |
| **Sequence patterns** | Often GC-rich, contain specific motifs recognized by proteins |

### Role in Gene Transcription

1. **Recognition**: Transcription factors (TFs) bind to specific motifs in the promoter
2. **Assembly**: The pre-initiation complex (PIC) forms at the promoter
3. **Initiation**: RNA Polymerase II begins synthesizing mRNA from the TSS
4. **Regulation**: Promoter strength determines basal transcription levels

### Why Promoter Detection Matters

- **Gene regulation research**: Understanding which sequences can initiate transcription
- **Genome annotation**: Identifying gene boundaries in newly sequenced genomes
- **Synthetic biology**: Designing promoters with desired expression levels
- **Disease research**: Mutations in promoters can cause aberrant gene expression

---

## The Dataset: InstaDeep Promoter Benchmark

We use the **promoter_all** task from InstaDeep's Nucleotide Transformer benchmark:

| Aspect | Details |
|--------|---------|
| **Source** | [InstaDeepAI/nucleotide_transformer_downstream_tasks](https://huggingface.co/datasets/InstaDeepAI/nucleotide_transformer_downstream_tasks) |
| **Input** | 300 bp DNA sequences centered around potential TSS |
| **Output** | Binary classification: promoter (1) or non-promoter (0) |
| **Positive samples** | Sequences containing active promoter regions |
| **Negative samples** | Random genomic sequences without promoter activity |

---

## Recap: What We've Learned

From Day 1:
- ✅ DNA encoding and tokenization (BPE, k-mer)
- ✅ Model architecture (Nucleotide Transformer)
- ✅ Embedding extraction
- ✅ Hugging Face ecosystem

Today we put it all together!

---

## Setup

In [1]:
import torch
import torch.nn as nn
import numpy as np
from collections import Counter
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    TrainingArguments, 
    Trainer
)
from datasets import load_dataset, DatasetDict
import evaluate
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Using device: {device}")

PyTorch version: 2.9.1+cu128
CUDA available: True
Using device: cuda


---

## Part 1: Load and Explore the Dataset

We'll use InstaDeep's benchmark dataset with 18 genomic tasks.

In [2]:
# Load the full dataset
print("Loading dataset from Hugging Face Hub...")
ds_all = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks")
print(ds_all)

Loading dataset from Hugging Face Hub...


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['sequence', 'name', 'label', 'task'],
        num_rows: 461850
    })
    test: Dataset({
        features: ['sequence', 'name', 'label', 'task'],
        num_rows: 48797
    })
})


In [3]:
# List available tasks
available_tasks = np.unique(ds_all['train']['task'])
print(f"Available tasks ({len(available_tasks)}):")
for task in available_tasks:
    print(f"  - {task}")

Available tasks (18):
  - H3
  - H3K14ac
  - H3K36me3
  - H3K4me1
  - H3K4me2
  - H3K4me3
  - H3K79me3
  - H3K9ac
  - H4
  - H4ac
  - enhancers
  - enhancers_types
  - promoter_all
  - promoter_no_tata
  - promoter_tata
  - splice_sites_acceptors
  - splice_sites_all
  - splice_sites_donors


In [4]:
# Select the promoter detection task
TASK = "promoter_all"  # Try others: "enhancers", "H3K4me3", "splice_sites_all"

# Filter to selected task
ds_full = ds_all.filter(lambda ex: ex["task"] == TASK)

# Create a subset for faster training (remove for full training)
ds = DatasetDict({
    "train": ds_full["train"].shuffle(seed=42).select(range(15000)),
    "test": ds_full["test"].shuffle(seed=42).select(range(5000))
})

print(f"Task: {TASK}")
print(f"Train samples: {len(ds['train']):,}")
print(f"Test samples: {len(ds['test']):,}")

Task: promoter_all
Train samples: 15,000
Test samples: 5,000


In [5]:
# Explore the data
print("Features:", ds['train'].features)

# Class distribution
labels = [ex['label'] for ex in ds['train']]
label_counts = Counter(labels)
print(f"\nClass distribution (training):")
for label, count in sorted(label_counts.items()):
    print(f"  Label {label}: {count:,} ({100*count/len(labels):.1f}%)")

# Sequence length
seq_lengths = [len(ex['sequence']) for ex in ds['train']]
print(f"\nSequence length: {min(seq_lengths)} - {max(seq_lengths)} bp")

# Sample
print(f"\nSample sequence (first 50bp): {ds['train'][0]['sequence'][:50]}...")

Features: {'sequence': Value('string'), 'name': Value('string'), 'label': Value('int32'), 'task': Value('string')}

Class distribution (training):
  Label 0: 7,531 (50.2%)
  Label 1: 7,469 (49.8%)

Sequence length: 300 - 300 bp

Sample sequence (first 50bp): AACAGGCTGAGAATAGTTTGACCTCTGATCACCAAAAAACCTGGAGGATA...


---

## Part 2: Tokenize the Dataset

Convert DNA strings to token IDs using the model's tokenizer.

In [6]:
# Model identifier - using the 500M Human Ref model (ESM-based, works reliably)
MODEL_ID = "InstaDeepAI/nucleotide-transformer-500m-human-ref"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
print(f"Model: {MODEL_ID}")
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")

# Max sequence length (in tokens)
MAX_LEN = 512

def preprocess(batch):
    """Tokenize DNA sequences and add labels."""
    tokens = tokenizer(
        batch["sequence"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    )
    tokens["labels"] = batch["label"]
    return tokens

# Apply tokenization
print("Tokenizing...")
tokenized = DatasetDict({
    k: v.map(preprocess, batched=True, remove_columns=v.column_names)
    for k, v in ds.items()
})

# Create validation split
split = tokenized["train"].train_test_split(test_size=0.1, seed=42)
tokenized = DatasetDict({
    "train": split["train"],
    "validation": split["test"],
    "test": tokenized["test"]
})

print(f"\nTokenized dataset:")
print(tokenized)

Model: InstaDeepAI/nucleotide-transformer-500m-human-ref
Tokenizer vocab size: 4107
Tokenizing...

Tokenized dataset:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 13500
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 5000
    })
})


---

## Part 3: Load the Model with Classification Head

Hugging Face provides `AutoModelForSequenceClassification` which:
1. Loads the pre-trained encoder
2. Automatically adds a classification head on top

### Architecture

```
Input IDs → NT Encoder → Pooler → Classification Head → Class Logits
            (pre-trained)         (randomly initialized)
```

The classification head weights are randomly initialized and will be trained along with fine-tuning the encoder.

In [7]:
# Number of classes for promoter detection
num_labels = len(set(ds["train"]["label"]))
print(f"Number of classes: {num_labels}")

# Load model with classification head
# AutoModelForSequenceClassification automatically adds a classification head
print(f"\nLoading {MODEL_ID} with classification head...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, 
    num_labels=num_labels,
    trust_remote_code=True
)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel loaded successfully!")
print(f"  Total parameters:     {total_params/1e6:.1f}M")
print(f"  Trainable parameters: {trainable_params/1e6:.1f}M")

Number of classes: 2

Loading InstaDeepAI/nucleotide-transformer-500m-human-ref with classification head...


Some weights of EsmForSequenceClassification were not initialized from the model checkpoint at InstaDeepAI/nucleotide-transformer-500m-human-ref and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Model loaded successfully!
  Total parameters:     480.4M
  Trainable parameters: 480.4M


---

## Part 4: Train the Model

We'll use the Hugging Face `Trainer` which handles:
- Batching and data loading
- Forward/backward passes
- Optimizer (AdamW)
- Learning rate scheduling
- Logging and evaluation

In [8]:
# Load evaluation metrics
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    """Compute accuracy and F1 for evaluation."""
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

In [9]:
# Training configuration
args = TrainingArguments(
    output_dir="nt_promoter_classifier",
    
    # Batch size (reduce if OOM)
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    
    # Learning rate
    learning_rate=2e-5,
    
    # Training duration
    num_train_epochs=2,
    
    # Evaluation
    eval_strategy="epoch",
    
    # Logging
    logging_steps=50,
    
    # Checkpointing
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    
    # Disable external logging
    report_to="none",
)

print("Training configuration:")
print(f"  Batch size: {args.per_device_train_batch_size}")
print(f"  Learning rate: {args.learning_rate}")
print(f"  Epochs: {args.num_train_epochs}")

Training configuration:
  Batch size: 8
  Learning rate: 2e-05
  Epochs: 2


In [10]:
# Create Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Train!
print("Starting training...")
print("="*50)
train_result = trainer.train()

# Print training results
print("\n" + "="*50)
print("Training complete!")
print(f"  Training loss: {train_result.training_loss:.4f}")

/tmp/ipykernel_2902506/1800702608.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.335200,0.254217,0.936000,0.935945
2,0.059200,0.238718,0.944667,0.944542



Training complete!
  Training loss: 0.1976


In [11]:
# Evaluate on validation set
print("Validation Results:")
print("="*50)
val_results = trainer.evaluate()
for k, v in val_results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

Validation Results:


  eval_loss: 0.2387
  eval_accuracy: 0.9447
  eval_f1_macro: 0.9445
  eval_runtime: 65.2595
  eval_samples_per_second: 22.9850
  eval_steps_per_second: 2.8810
  epoch: 2.0000


---

## Part 5: Evaluate on Test Set

Final evaluation on held-out test data.

In [ ]:
# Predict on test set
print("Evaluating on test set...")
predictions = trainer.predict(tokenized["test"])

y_true = predictions.label_ids
y_pred = predictions.predictions.argmax(-1)

# Classification report
print("\n" + "="*50)
print("Classification Report")
print("="*50)
print(classification_report(y_true, y_pred, digits=4))

Evaluating on test set...


KeyboardInterrupt: 

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

print("Confusion Matrix:")
print(cm)
print(f"\n  Rows = true labels, Columns = predicted labels")
print(f"  Correct predictions: {cm.diagonal().sum()}/{len(y_true)} ({100*cm.diagonal().sum()/len(y_true):.1f}%)")

# Plot confusion matrix
plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'Confusion Matrix - {TASK}')
for i in range(len(cm)):
    for j in range(len(cm)):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center')
plt.tight_layout()
plt.show()

---

## Exercises: Experiment!

### 1. Try a Different Task
Change `TASK = "promoter_all"` to:
- `"enhancers"` - Enhancer detection
- `"H3K4me3"` - Histone modification
- `"splice_sites_all"` - Splice site prediction

### 2. Tune Hyperparameters
- Learning rate: `1e-5`, `5e-5`, `3e-4`
- Batch size: `8`, `32`, `64`
- Epochs: `3`, `5`, `10`

### 3. Try a Larger Model
- `InstaDeepAI/nucleotide-transformer-500m-human-ref`

### 4. Freeze the Encoder
Only train the classification head:
```python
for param in model.base_model.parameters():
    param.requires_grad = False
```

---

## Summary

In this notebook, you learned the complete fine-tuning workflow:

| Step | What You Did |
|------|-------------|
| 1. Data | Loaded and explored benchmark dataset |
| 2. Tokenize | Converted DNA to token IDs |
| 3. Model | Built classifier with pre-trained backbone |
| 4. Train | Used HF Trainer for training loop |
| 5. Evaluate | Assessed with accuracy, F1, confusion matrix |

**Next**: We'll learn **PEFT/LoRA** for parameter-efficient fine-tuning (training only ~1% of parameters)!